# Foraward simulation for sandbox

In [1]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## test

In [7]:
import pandas as pd


class SLE:

    def __init__(self):

        # 井資料庫
        # self.wells = []

        # 範例 node table
        self.node = pd.DataFrame({
            "node_id": [13, 14, 8, 10],
            "x": [100, 200, 300, 250],
            "y": [50, 50, 100, 80]
        })

    def select_nodes(self, by, value):

        if by.lower() != "coordinate":
            raise ValueError("Only coordinate selection is implemented.")

        x, y = value

        return self.node[
            (self.node["x"] == x) &
            (self.node["y"] == y)
        ]

    def add_wells(self, *wells):
        """
        Add wells.

        Parameters
        ----------
        w1 :
            (
                stress,
                coordinate,
                name,
                type,
                parameter(dict)
            )
        wells = (w1 w2, ...)
        
        """
        self.wells = []
        for well in wells:

            stress, coordinate, name, well_type, parameter = well

            df_node = self.select_nodes(
                by="coordinate",
                value = coordinate
            )

            if df_node.empty:
                raise ValueError(
                    f"Cannot find node at {coordinate}"
                )

            node_id = int(df_node.iloc[0]["node_id"])

            self.wells.append({
                "stress": stress,
                "node_id": node_id,
                "coordinate": coordinate,
                "name": name,
                "type": well_type.lower(),
                "parameter": parameter
            })

        return self

    
    def create_observation(self):

        rows = []

        # by "observation" 
        stress_list = sorted(
            {w["stress"] for w in self.wells
             if w["type"] == "observation"}
        )
        
        for stress_idx in stress_list:
            
            # by "stress"
            obs = [
                w for w in self.wells
                if w["stress"] == stress_idx 
                and w["type"] == "observation"
            ]

            rows.append([stress_idx, len(obs)])

            for well in obs:

                rows.append([ well["node_id"], well["name"]])

        return pd.DataFrame(rows)


    def create_source(self):


        # select by well type
        stress_list = sorted(
            {w["stress"] for w in self.wells
             if w["type"] == "injection"}
        )
        
        df_up = pd.DataFrame([[len(stress_list)], [0]]) # for [total stress number] and [0]
        
        src_list = [df_up]
        # store each stress
        for stress_idx in stress_list:

            src = [
                w for w in self.wells
                if w["stress"] == stress_idx and  w["type"] == "injection"
            ]

            rows = []
            df_between = pd.DataFrame([len(src)])

            for well in src:

                rows.append([
                    well["node_id"],              # node idex
                    well["parameter"]["rate"],    # sink/source for flow
                    0,                            # sink/source for concentration
                    well["parameter"]["time"][0], # start time
                    well["parameter"]["time"][1], # end time
                    0,                            # start time for concentration
                    0,                            # end time for concentration
                    well["name"],                 # well name
                ])
                
            df = pd.DataFrame(rows)
            src_list.append(pd.concat([df_between, df]))
            
        
        return pd.concat(src_list)

 
    def create_times(self, *times):
        """ 
        Add time.

        Parameters
        ----------
        t1 =  (dt, dt_max, dt_multipler, t_max, output_flag, t_reduction)
        
        """
        t_list = []
        for t_ in times:
            dt, dt_max, dt_m, t_max, flag, t_re = t_
            
            if flag == 'time':
                flag_value = 0
            
            t_list.append([
                dt, 
                dt_max, 
                dt_m,
                t_max,
                flag_value,
                t_re
            ])
        
        
        return pd.DataFrame(t_list).T   
            
        
        
    


In [8]:
model = SLE()

In [9]:
ob1 = ( 1, (100, 50), "OBS-1", "observation", {} )
ob2 = ( 1, (100, 50), "OBS-2", "observation", {} )
ob3 = ( 2, (100, 50), "OBS-3", "observation", {} )
inj1 = ( 1, (100, 50), "INJ_1", "injection", {'rate': 10, 'time': (0, 20)} )
inj2 = ( 2, (100, 50), "INJ_2", "injection", {'rate': 10, 'time': (0, 20)} )
inj3 = ( 3, (100, 50), "INJ_3", "injection", {'rate': 10, 'time': (0, 20)} )

model.add_wells(
    ob1,
    ob2,
    ob3,
    inj1,
    inj2,
    inj3,
    
)

print(model.wells)
print(model.create_observation())
print(model.create_source())


# t_1 = (5, 10, 1, 600, 'time', 0)      
# t_2 = (10, 10, 1, 600, 'time', 0)
# t_3 = (15, 10, 1, 600, 'time', 0)    

# print(model.create_times(t_1, t_2, t_3))



[{'stress': 1, 'node_id': 13, 'coordinate': (100, 50), 'name': 'OBS-1', 'type': 'observation', 'parameter': {}}, {'stress': 1, 'node_id': 13, 'coordinate': (100, 50), 'name': 'OBS-2', 'type': 'observation', 'parameter': {}}, {'stress': 2, 'node_id': 13, 'coordinate': (100, 50), 'name': 'OBS-3', 'type': 'observation', 'parameter': {}}, {'stress': 1, 'node_id': 13, 'coordinate': (100, 50), 'name': 'INJ_1', 'type': 'injection', 'parameter': {'rate': 10, 'time': (0, 20)}}, {'stress': 2, 'node_id': 13, 'coordinate': (100, 50), 'name': 'INJ_2', 'type': 'injection', 'parameter': {'rate': 10, 'time': (0, 20)}}, {'stress': 3, 'node_id': 13, 'coordinate': (100, 50), 'name': 'INJ_3', 'type': 'injection', 'parameter': {'rate': 10, 'time': (0, 20)}}]
    0      1
0   1      2
1  13  OBS-1
2  13  OBS-2
3   2      1
4  13  OBS-3
    0     1    2    3     4    5    6      7
0   3   NaN  NaN  NaN   NaN  NaN  NaN    NaN
1   0   NaN  NaN  NaN   NaN  NaN  NaN    NaN
0   1   NaN  NaN  NaN   NaN  NaN  NaN  

## Simulation control (User input)

In [2]:
# simulation control
project_name = 'test_0701'
simulation_control = ('3D', 'steady', 'confined')  # dimension, problem, aquifer

# Geometry control
dx = np.array( [43.5, 25, 20, 20, 20] + [7.5] * 19 + [6, 6, 5.5, 5.5, 6, 6] + [7.5] * 19 + [20, 20, 20, 25, 43.5])
dy = np.array( [31, 20, 20, 20, 10] + [7.5] * 24 + [10, 20, 20, 20, 31])
dz = np.array( [30, 25, 20, 15, 10, 10] + [5] * 11 + [7.5, 7.5] + [10, 10, 10, 25, 30, 35] )

start_coord = (0, 0, 0)         # origin coordinate (x0, y0, z0)
element_num = (54, 34, 25)      # element number in each direction (e_x, e_y, e_z)
element_spacing = (dx, dy, dz)  # element spacing in each direction (len(dx) = e_x-)

# initial condition
init_paras = (350, 0, 0.004, 0.00001, 0.4) # init_h, init_flux, init_K, init_Ss, porosity

# boundary condition
boundary = ('surface', 'UP', 'head')  # method, value, bc_type, bc_value

## Used loop to create dict in future??
# wells (stress_idx, coordinate, well_name, role, *{rate: ..., time: ...})

# sink/sources well
const_rate = 600
inj_time = (0, 600)
inj_1 = (1, ((288.5, 71,  120),), 'inj_1', 'injection', {'rate':const_rate, 'time': inj_time})
inj_2 = (2, ((88.5,  71,  110),), 'inj_2', 'injection', {'rate':const_rate, 'time': inj_time})
inj_3 = (3, ((88.5,  191, 100),), 'inj_3', 'injection', {'rate':const_rate, 'time': inj_time})
inj_4 = (4, ((88.5,  311, 90 ),), 'inj_4', 'injection', {'rate':const_rate, 'time': inj_time})
inj_5 = (5, ((288.5, 311, 120),), 'inj_5', 'injection', {'rate':const_rate, 'time': inj_time})
inj_6 = (6, ((488.5, 311, 110),), 'inj_6', 'injection', {'rate':const_rate, 'time': inj_time})
inj_7 = (7, ((488.5, 191, 100),), 'inj_7', 'injection', {'rate':const_rate, 'time': inj_time})
inj_8 = (8, ((488.5, 71,  90 ),), 'inj_8', 'injection', {'rate':const_rate, 'time': inj_time})

# observation well
ob_1 = (1, ((188.5, 71,  130),), 'Pie_1', 'observation', {})
ob_2 = (2, ((88.5,  131, 200),), 'Pie_2', 'observation', {})
ob_3 = (3, ((88.5,  251, 180),), 'Pie_3', 'observation', {})
ob_4 = (4, ((188.5, 311, 150),), 'Pie_4', 'observation', {})
ob_5 = (5, ((388.5, 311, 150),), 'Pie_5', 'observation', {})
ob_6 = (6, ((488.5, 251, 180),), 'Pie_6', 'observation', {})
ob_7 = (7, ((488.5, 131, 200),), 'Pie_7', 'observation', {})
ob_8 = (8, ((388.5, 71,  130),), 'Pie_8', 'observation', {})


# time info for transient (dt, dt_max, dt_mul, t_max, _, _)
t_1 = (10, 10, 1, 600, 'each_time_step', 0)      
t_2 = (10, 10, 1, 600, 'each_time_step', 0)
t_3 = (10, 10, 1, 600, 'each_time_step', 0)      
t_4 = (10, 10, 1, 600, 'each_time_step', 0)      
t_5 = (10, 10, 1, 600, 'each_time_step', 0)      
t_6 = (10, 10, 1, 600, 'each_time_step', 0)      
t_7 = (10, 10, 1, 600, 'each_time_step', 0)      
t_8 = (10, 10, 1, 600, 'each_time_step', 0)                                           
                                        

## Import sle_io package

In [3]:
from sleio import sle_io
test_forward = sle_io(project_name)
test_forward.set_parameters(simulation_control)
test_forward.create_geometry(start_coord, element_num, element_spacing)
test_forward.create_initial(init_paras)
test_forward.create_boundary(boundary)
test_forward.create_times(t_1, t_2, t_3, t_5, t_6, t_7, t_8)
test_forward.add_wells(ob_1, ob_2, ob_3, ob_4, ob_5, ob_6, ob_7, ob_8, 
                       inj_1, inj_2, inj_3, inj_4, inj_5, inj_6, inj_7, inj_8 )
test_forward.create_source()
test_forward.create_observation()
test_forward.create_function()
test_forward.create_simulation()
test_forward.create_problem()
test_forward.write_forward_input(path = 'test_folder')

Project: test_0701
Simulation with '3D' case, 'confined' aquifer under 'steady' state
Successfully wrote 12 files to:
C:\Users\Ethan\Desktop\ethan\專案\HTNN_Sandbox\test_folder
Ready for forward simulation!


## test forward simulation

In [12]:
#%% test forward simulation (try previous sle result)
# os.chdir(os.path.join(proj_dir, 'inverse_after_clogging'))
# df_result_lnK_mean, df_result_lnK_var = parse_estimation('O-kestimate.dat', dim =3)
os.chdir(os.path.join(proj_dir, 'inverse'))
df_result = pd.read_csv('HTNN_result.csv')

test_list = []
for i , well in enumerate(well_name): 
    os.chdir(os.path.join(work_dir, proj_name, 'forward', well))
    sle_forward_htnn = sleio(f'test_forward with {well} injection')
    k_est =  np.exp(df_result['HTNN_after_clogging'].values)
    df = sle_forward_htnn.run_forward( k_est, 0.00025, 'steady', init_h , plot_head = False)
    test_list.append(df.values[0])
simulation_head = np.concatenate(test_list, axis = 0)
# print(simulation_head.shape)

NameError: name 'proj_dir' is not defined

### Create multiple forward 

In [ ]:
# write forward files
inj_ws = [
    'inj_1','inj_2','inj_3','inj_4','inj_5','inj_6','inj_7','inj_8',
]

for i in range(len(inj_ws)):

    wirting_dir = f'\forward {inj_ws[i]}'
    if 
        os.mkdir()
    
    else:
    
    test_forward = sle_io(project_name)
    test_forward.set_parameters(simulation_control)
    test_forward.add_geometry(start_coord, element_num, element_spacing)
    test_forward.add_initial(init_paras)